# ShiftGuard-X — Notebook N119 (reconstructed)
## Arithmetic-path selection: information-cost feasibility audit

**Reconstructed from the user-provided NB118 results ZIP, not recovered original N119.** This notebook does not claim codec novelty or measured compression superiority.

**Research question:** Are the extra exact target matches from alternative floating-point addition paths potentially worth their selection/exception overhead? We evaluate information-theoretic lower bounds, not a production codec.

**Reproducibility:** Run all cells in Google Colab. Upload `SGX_NB118_results(1).zip` when prompted. No internet required.

In [ ]:
import json, zipfile, io, csv, math, pathlib, hashlib, platform, sys
from google.colab import files
uploaded = files.upload()
assert len(uploaded)==1, 'Upload only the NB118 results ZIP'
filename, blob = next(iter(uploaded.items()))
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    required={'decision.json','comparison.csv','real_source_results.json','README.md'}
    assert required.issubset(set(z.namelist())), 'Missing NB118 result files'
    decision=json.loads(z.read('decision.json'))
    comparison=list(csv.DictReader(io.StringIO(z.read('comparison.csv').decode())))
    source=json.loads(z.read('real_source_results.json'))
print('ZIP SHA256:',hashlib.sha256(blob).hexdigest())
print('NB118 decision:',decision)
assert decision['notebook']=='NB118' and decision['all_three_real_sources_available']
assert len(comparison)==3

## 1. Reproduce NB118 prevalence counts
The **extra exact matches** are the rows where a nonbaseline path equals the recorded target and the baseline path does not. These are *not* saved bits. The count says nothing about how many bits it costs to signal the path, identify exceptions, or store remaining residuals.

In [ ]:
import pandas as pd
df=pd.DataFrame(comparison)
for col in df.columns:
    if col!='dataset': df[col]=pd.to_numeric(df[col])
df['recomputed_extra_rate']=df.additional_exact_target_by_other_path/df.qualified_rows
assert (abs(df.recomputed_extra_rate-df.additional_exact_fraction)<1e-12).all()
display(df[['dataset','qualified_rows','p0_exact_target','additional_exact_target_by_other_path','recomputed_extra_rate','unexplained_by_any_path']])

## 2. Binary selection lower bound
Even if an oracle tells us whether an alternative is needed, losslessly encoding the binary selection pattern across rows requires approximately **N × H₂(p)** bits under an i.i.d. Bernoulli model with optimal entropy coding, where p is the extra-match rate. This is a *model-based entropy estimate*, not a universal lower bound on every structured sequence: context or side information can reduce the conditional entropy. A fixed path ID for one of five paths would additionally cost up to ceil(log₂ 5)=3 bits per explicitly selected row without more efficient coding.

**Important:** These estimates ignore correlations and path-ID distributions; they are only screening calculations.

In [ ]:
def h2(p):
    return 0.0 if p in (0,1) else -p*math.log2(p)-(1-p)*math.log2(1-p)
rows=[]
for r in df.itertuples():
    p=r.recomputed_extra_rate; n=int(r.qualified_rows); extra=int(r.additional_exact_target_by_other_path)
    rows.append(dict(dataset=r.dataset, extra_rows=extra, extra_rate=p, binary_flag_entropy_bits_per_row=h2(p), binary_flag_estimated_bits=n*h2(p), binary_flag_bits_per_extra_row=h2(p)/p, naive_3bit_path_id_bits=3*extra))
fees=pd.DataFrame(rows)
display(fees.round(4))

## 3. Break-even residual savings sensitivity
Let **S** be the average bits avoided for each additional exact row, compared with a specified residual coder. A necessary screening inequality for a naive per-row flag plus 3-bit alternative-path identifier is: 

`S > H₂(p)/p + 3`

This is **not sufficient** for whole-file compression: base predictions, residuals, null masks, framing, and metadata also have costs. We vary hypothetical S rather than inventing measured residual bit lengths.

In [ ]:
savings=[4,8,12,16,24,32,64]
sensitivity=[]
for r in fees.itertuples():
    threshold=r.binary_flag_bits_per_extra_row+3
    for s in savings:
        sensitivity.append(dict(dataset=r.dataset, assumed_bits_saved_per_extra=s, necessary_threshold_bits_per_extra=threshold, screen_pass=s>threshold, modeled_net_bits=(s-threshold)*r.extra_rows))
sens=pd.DataFrame(sensitivity)
display(sens.pivot(index='assumed_bits_saved_per_extra',columns='dataset',values='modeled_net_bits').round(1))
print('These are hypothetical modeled bits, NOT measured compressed sizes.')

## 4. Conservative prior-art and novelty audit
Multi-predictor selection, exception coding, floating-point compression, and dependency-aware encoding are established research areas. Names to check carefully include **FPC**, **ALP**, **Corra**, and **Virtual**; verify original papers and their exact mechanisms before any novelty statement.

A plausible *hypothesis* to investigate is whether deterministic IEEE-754 arithmetic-path equivalence can be represented with a lower-cost context-conditioned selector than generic residual encoding. This notebook does **not** prove that hypothesis.

To test it properly, N120 must obtain row-level data, train selection/context models only on training data, test on held-out months, implement an actual bitstream and exact decoder, and compare total file sizes and runtimes to baselines. Avoid training on the recorded target at decode time.

In [ ]:
novelty_audit=pd.DataFrame([
 {'claim':'Alternative arithmetic orders yield additional exact matches','status':'SUPPORTED by NB118 prevalence counts','evidence':'Three source-level counts'},
 {'claim':'Path selector can be inferred at decoder without target','status':'UNTESTED','evidence':'Requires reproducible decoder-visible features'},
 {'claim':'Selector overhead is below residual savings','status':'UNTESTED','evidence':'Needs actual residual and bitstream coding'},
 {'claim':'Whole-file lossless round trip','status':'UNTESTED','evidence':'Requires encoder, decoder, bit-exact checks'},
 {'claim':'Beats existing codecs on held-out files','status':'UNTESTED','evidence':'Requires apples-to-apples byte counts and timing'},
 {'claim':'Algorithm is distinct from prior art','status':'UNESTABLISHED','evidence':'Requires primary-paper comparison'}])
display(novelty_audit)

## 5. Export reproducible N119 checkpoint
The JSON/CSV output distinguishes NB118 measured prevalence from N119 modeled overhead. No synthetic savings are presented as empirical codec performance.

In [ ]:
from pathlib import Path
out=Path('SGX_NB119_results'); out.mkdir(exist_ok=True)
fees.to_csv(out/'selection_entropy_screen.csv',index=False)
sens.to_csv(out/'hypothetical_savings_sensitivity.csv',index=False)
novelty_audit.to_csv(out/'novelty_evidence_audit.csv',index=False)
report={'notebook':'NB119_reconstructed','source_notebook':'NB118','source_zip_sha256':hashlib.sha256(blob).hexdigest(),'measured_codec_gain':None,'novel_algorithm_established':False,'selection_screen_is_model_based':True,'next_step':'N120: real row-level decoder-visible selector and bit-exact codec benchmarking'}
(out/'decision.json').write_text(json.dumps(report,indent=2))
(out/'README.md').write_text('N119 reconstructed information-cost feasibility audit. Only NB118 prevalence is measured; all savings sensitivity values are hypothetical. No codec benchmark or novelty established.\n')
import shutil
archive=shutil.make_archive('SGX_NB119_results','zip',root_dir='.',base_dir=str(out))
print('Created',archive)
files.download(archive)